In [1]:
print("Hello, world")

Hello, world


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DepthFromMotionNet(nn.Module):
    """
    Input:
        x: [batch, T, H, W]
           T consecutive retinal frames stacked along channel dimension

    Output:
        logits: [batch, 2]
           class 0 = left dot is closer
           class 1 = right dot is closer
    """
    def __init__(self, num_frames=5, hidden_dim=32):
        super().__init__()

        # Early visual / SC-like feature layer
        self.conv1 = nn.Conv2d(
            in_channels=num_frames,
            out_channels=16,
            kernel_size=5,
            padding=2
        )

        self.conv2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        # Small decoder
        self.fc1 = nn.Linear(32 * 16 * 16, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        # x: [B, T, H, W]
        x = F.relu(self.conv1(x))      # [B, 16, H, W]
        x = F.max_pool2d(x, 2)         # [B, 16, H/2, W/2]

        x = F.relu(self.conv2(x))      # [B, 32, H/2, W/2]
        x = F.max_pool2d(x, 2)         # [B, 32, H/4, W/4]

        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)
        return logits

ModuleNotFoundError: No module named 'torch'

In [ ]:
import torch
from torch.utils.data import Dataset
import random


class TwoDotDepthDataset(Dataset):
    """
    Generates simple 5-frame movies with:
    - one dot on the left
    - one dot on the right
    - nearer dot moves faster
    Label:
        0 = left dot is closer
        1 = right dot is closer
    """
    def __init__(self, n_samples=1000, T=5, H=64, W=64):
        self.n_samples = n_samples
        self.T = T
        self.H = H
        self.W = W

    def __len__(self):
        return self.n_samples

    def draw_dot(self, frame, x, y, radius=2, value=1.0):
        for i in range(max(0, y - radius), min(self.H, y + radius + 1)):
            for j in range(max(0, x - radius), min(self.W, x + radius + 1)):
                if (i - y) ** 2 + (j - x) ** 2 <= radius ** 2:
                    frame[i, j] = value

    def __getitem__(self, idx):
        movie = torch.zeros(self.T, self.H, self.W)

        # Fixed vertical positions with a little noise
        y_left = random.randint(18, 46)
        y_right = random.randint(18, 46)

        # Initial x positions
        x_left0 = random.randint(8, 20)
        x_right0 = random.randint(44, 56)

        # Choose which dot is closer
        left_is_closer = random.choice([True, False])

        # Nearer object moves faster across retina
        fast_speed = random.randint(3, 5)
        slow_speed = random.randint(1, 2)

        v_left = fast_speed if left_is_closer else slow_speed
        v_right = slow_speed if left_is_closer else fast_speed

        # Move both dots horizontally
        for t in range(self.T):
            frame = torch.zeros(self.H, self.W)

            x_left = x_left0 + t * v_left
            x_right = x_right0 - t * v_right

            self.draw_dot(frame, x_left, y_left)
            self.draw_dot(frame, x_right, y_right)

            movie[t] = frame

        label = 0 if left_is_closer else 1
        return movie, torch.tensor(label, dtype=torch.long)

In [ ]:
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds = TwoDotDepthDataset(n_samples=2000)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

model = DepthFromMotionNet(num_frames=5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == y).sum().item()
        total += x.size(0)

    print(f"Epoch {epoch+1}: loss={total_loss/total:.4f}, acc={total_correct/total:.4f}")